# Federated hybrid MLP/LightGBM — ICU mortality across 147 real hospitals

WiDS Datathon 2020 / GOSSIS. 91,713 ICU stays, `hospital_death` at 8.63%
prevalence, partitioned by the real `hospital_id` rather than a simulated split.

Runs both split regimes, which answer different questions:

* **within-site** (`random`) — hospitals appear in train and test. Isolates the
  *cost* of federation, since every model sees identically distributed test data.
* **cross-site** (`site_holdout`) — whole hospitals held out. Measures what
  federation *buys*: generalisation to an institution never seen in training.

Attach: `fedlgbm-src` (Your Datasets) and the **WiDS Datathon 2020** competition.
Run with **Save Version → Save & Run All (Commit)**; an interactive session
times out first.

**Accelerator:** prefer *GPU T4 x2* (sm_75). Kaggle's P100 is sm_60 and this
torch build ships no sm_60 kernels, so the preflight below detects it and falls
back to CPU rather than failing. CPU is workable for this dataset — it is 91k x
393, an order of magnitude smaller than the CIC-IoT corpus — but a T4 is faster.
The resolved device is recorded in every `config.json`, and all models in a run
use the same one, since results are not bit-identical across devices.

In [ ]:
import sys, time, json, shutil
from pathlib import Path

INPUT = Path('/kaggle/input')
OUT = Path('/kaggle/working/outputs')
print('attached:', [p.name for p in INPUT.iterdir()] if INPUT.exists() else 'NOTHING')

# Locate rather than assume: Kaggle's extraction does not always preserve the
# directory level the local tree had.
hits = sorted(INPUT.glob('*/**/fedlgbm/__init__.py'))
assert hits, 'fedlgbm not found — attach the fedlgbm-src dataset'
SRC = hits[0].parent.parent
sys.path.insert(0, str(SRC))
print('source root:', SRC)

train_files = sorted(INPUT.glob('**/training_v2.csv'))
assert train_files, ('training_v2.csv not found — attach the WiDS Datathon 2020 '
                     'competition via Add Data -> Competitions')
DATA = train_files[0].parent
print('data:', train_files[0], f'({train_files[0].stat().st_size/1e6:.1f} MB)')

In [ ]:
import torch

# A P100 (sm_60) with a cu12x torch build reports cuda.is_available() as True and
# then fails every kernel launch with "no kernel image is available". Detect it
# by launching a real kernel, not by trusting the availability flag.
#
# Unlike the CIC-IoT sweeps, falling back to CPU is acceptable here: this dataset
# is 91k x 393 against 754k x 39, so a full run is well inside Kaggle's limit.
# What is NOT acceptable is running half the models on one device and half on
# another, so the resolved device is recorded and used for everything.
DEVICE = 'cpu'
GPU_NOTE = 'no GPU attached'

if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    name = torch.cuda.get_device_name(0)
    print(f'gpu: {name}  sm_{cap[0]}{cap[1]}')
    print(f'torch {torch.__version__} built for: {torch.cuda.get_arch_list()}')
    try:
        (torch.zeros(8, 8, device='cuda') @ torch.ones(8, 8, device='cuda')).sum().item()
        DEVICE, GPU_NOTE = 'cuda', f'{name} (sm_{cap[0]}{cap[1]})'
        print('CUDA preflight passed')
    except Exception as exc:
        GPU_NOTE = (f'{name} sm_{cap[0]}{cap[1]} unusable with torch '
                    f'{torch.__version__}: {type(exc).__name__}')
        print(f'\n*** CUDA UNUSABLE — falling back to CPU ***\n{GPU_NOTE}\n'
              'For a faster run: Settings -> Accelerator -> "GPU T4 x2" (sm_75), '
              'then Save & Run All again.\n')

print(f'device = {DEVICE}   ({GPU_NOTE})')

In [ ]:
from fedlgbm import ExperimentConfig, setup_logging
from fedlgbm.experiments import run_all

setup_logging()

def build(split_mode: str, name: str) -> ExperimentConfig:
    cfg = ExperimentConfig()
    cfg.name = name
    cfg.output_dir = OUT
    cfg.device = DEVICE
    cfg.skip_privacy_attacks = False   # inversion/MIA are a headline result here

    cfg.data.domain = 'wids'
    cfg.data.data_dir = DATA
    cfg.data.file_glob = 'training_v2.csv'
    cfg.data.max_rows = None
    cfg.data.max_per_class = None      # capping would destroy the 8.63% base rate
    cfg.data.split_mode = split_mode

    # One client per real hospital; count is an output, not a setting.
    cfg.partition.scheme = 'natural'
    cfg.partition.min_samples_per_client = 200

    cfg.fed.n_rounds = 30
    cfg.fed.local_epochs = 2
    cfg.mlp.hidden_dims = (256, 128)
    cfg.mlp.embedding_dim = 64

    # class_weight=None keeps probabilities calibrated. Balanced weighting buys
    # recall at 0.5 by destroying calibration, and Brier/slope are reported.
    cfg.lgbm.class_weight = None
    cfg.lgbm.num_leaves = 255
    cfg.lgbm.n_estimators = 1000
    return cfg

SKIP = {'lstm', 'isolation_forest'}   # neither is meaningful on tabular ICU rows
print('configured')

In [ ]:
results = {}
for mode, name in (('random', 'wids_within_site'),
                   ('site_holdout', 'wids_cross_site')):
    print('=' * 70, f'\n{name}  (split_mode={mode})\n', '=' * 70)
    t0 = time.time()
    results[name] = run_all(build(mode, name), skip=SKIP)
    print(f'{name}: {(time.time() - t0) / 60:.1f} min')

In [ ]:
from fedlgbm import metrics

for name, res in results.items():
    print('\n' + '=' * 70, f'\n{name}\n', '=' * 70)
    print(metrics.comparison_table(res['models']))
    d = res['dataset']['clean_stats']['split']
    print(f"\nsplit: {d['n_train']}/{d['n_val']}/{d['n_test']}  "
          f"prevalence {d['prevalence_train']:.4f}/{d['prevalence_val']:.4f}/"
          f"{d['prevalence_test']:.4f}  sites spanning: {d.get('n_sites_spanning_splits')}")
    print(f"clients: {res['partition']['n_clients']}  "
          f"label skew: {res['partition'].get('mean_label_skew')}")
    if 'privacy_attacks' in res:
        print('privacy:', json.dumps(res['privacy_attacks'])[:400])

shutil.make_archive('/kaggle/working/wids_outputs', 'zip', OUT)
for name in results:
    src = OUT / name / 'comparison_table.md'
    if src.exists():
        shutil.copy2(src, Path('/kaggle/working') / f'{name}_table.md')
print('\narchived')